# VC-SASR 본 평가 벤치마크

본 평가셋 두 벌(`eval-t2` 주 조건, `eval-t2-static` 정지 조건)에서 모든 시스템을 같은 채점기로 잽니다. 이 노트북은 **아무것도 학습하지 않습니다.** 고정 조건(평가셋, 전사기, 화자 임베딩, 채점 규칙)은 [BENCHMARK.md](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/BENCHMARK.md)에 있습니다.

| `SYSTEMS` | 실행하는 것 |
|---|---|
| `all` (기본값) | 아래 `ours`와 `mono`를 순서대로 |
| `ours` | 우리 시스템 6종: 비교군 3종(`voice_only`, `space_only`, `naive_fusion`) + 제안 모델 3종(`proposed_carry`, `proposed_reset`, `proposed_carry_nodrop`) |
| `mono` | 최신 단일채널 화자 구분 비교군 3종: Nemotron + Qwen3-ASR, VibeVoice-ASR, MOSS |
| `track_b` | Track B: VibeVoice-ASR만 |
| 시스템 이름 하나 | 그 시스템만 (예: `proposed_carry`, `vibevoice`) |

- **우리 시스템**은 학습 노트북([`train_template.ipynb`](https://colab.research.google.com/github/hd1534/variable-channel-spatial-asr/blob/main/notebooks/train_template.ipynb))이 같은 개인 Drive 백업에 남긴 `runs/<시스템>`(검증에서 고른 임계값과 `best.pt`)을 읽습니다. 학습·검증이 아직 끝나지 않은 시스템은 건너뛰고 마지막에 목록으로 알려 줍니다. 학습 노트북이 다른 런타임에서 돌고 있어도 이 노트북을 같은 `DRIVE_AUTH`로 연결할 수 있고, 다시 실행하면 그사이 끝난 시스템만 새로 채점합니다.
  - 전사기는 `ASR="all"`이면 **Qwen3-ASR-1.7B**(`qwen3-1.7b`)와 `wav2vec2` 두 가지로 각각 채점해 `eval/<실행>-<전사기>/`에 따로 둡니다.
  - 화자·공간 임베딩(`embeddings.npz`)은 시스템마다 한 번만 계산해 두 전사기가 함께 씁니다.
- **최신 단일채널 비교군**의 실제 입력은 항상 마이크 하나입니다. 우리 시스템과 같은 기준 채널(활성 마이크 중 최소 번호)을 쓰고, 결과는 `mono_baselines/<모델>/`에 둡니다.
  - 20초 전체를 보는 `full_audio` 참고 행이라 공식 `causal` 표와 따로 보고합니다.
  - VibeVoice는 가중치만 약 18GB라 T4에서는 돌지 않습니다. 단일채널 비교군을 돌릴 때에는 A100을 고르세요.
  - `MONO_LIMIT=2`로 두면 세션 두 개로 추론·재개·채점만 먼저 확인합니다. `0`이면 전체를 돌리고, 끝난 세션은 건너뜁니다.
- **이미 끝난 결과는 건너뜁니다.** 같은 `summary.json`·`best.pt`·전사기·전사 캐시로 채점한 결과가 있으면 다시 돌리지 않습니다.

## 0. 설정

In [ ]:
STORAGE_BACKEND: str = "drive_api"  # @param ["drive_api", "mounted_drive"]
DRIVE_SECRET: str = "DRIVE_AUTH"  # @param {type:"string"}
LOCAL_STORE_ROOT: str = "/content/vcsasr-store"
BACKUP_EVERY_SEC: int = 120  # @param {type:"integer"}
# 이 VM으로 내려받지 않을 백업 경로 접두어 목록. None이면 기본값(이 실행에서 쓰지 않는 경로)을 쓰고 []이면 모두 받습니다.
# 건너뛴 파일도 백업에는 그대로 남습니다.
RESTORE_SKIP: list[str] | None = None

SYSTEMS: str = "all"  # @param ["all", "ours", "mono", "track_b", "voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop", "nemotron", "vibevoice", "moss"]
ASR: str = "all"  # @param ["all", "qwen3-1.7b", "wav2vec2"]
# 학습 노트북에서 실행 이름에 붙은 꼬리표. 기본 학습이면 비워 둡니다.
# 예: SPEAKERS="dev-clean-only"면 "-devclean", RUN_TAG="x"면 "-x", 제안 모델의 train.max_steps를 바꿨으면 "-steps2000"
RUN_SUFFIX: str = ""  # @param {type:"string"}
MONO_LIMIT: int = 0  # @param {type:"integer"}
MONO_MAX_NEW_TOKENS: int = 2048  # @param {type:"integer"}
MONO_RUN_TAG: str = ""  # @param {type:"string"}
ASR_CACHE_NAME: str = "asr_cache"  # @param {type:"string"}
BRANCH: str = "main"  # @param {type:"string"}
COMMIT: str = ""  # @param {type:"string"}

REPO_URL: str = "https://github.com/hd1534/variable-channel-spatial-asr.git"
REPO_DIR: str = "/content/variable-channel-spatial-asr"
DRIVE_ROOT: str = LOCAL_STORE_ROOT if STORAGE_BACKEND == "drive_api" else "/content/drive/MyDrive/vcsasr"
ENV_ROOT: str = "/content/vcsasr-envs"
SETS: list[str] = ["eval-t2", "eval-t2-static"]

OURS: list[str] = ["voice_only", "space_only", "naive_fusion", "proposed_carry", "proposed_reset", "proposed_carry_nodrop"]
MONO: list[str] = ["nemotron", "vibevoice", "moss"]
PRESETS: dict[str, list[str]] = {"all": OURS + MONO, "ours": OURS, "mono": MONO, "track_b": ["vibevoice"]}
SELECTED: list[str] = PRESETS.get(SYSTEMS, [SYSTEMS])
if any(name not in OURS + MONO for name in SELECTED):
    raise ValueError(f"SYSTEMS를 확인하세요: {SYSTEMS}")
OURS_SELECTED: list[str] = [name for name in SELECTED if name in OURS]
MONO_SELECTED: list[str] = [name for name in SELECTED if name in MONO]
ASR_LIST: list[str] = ["qwen3-1.7b", "wav2vec2"] if ASR == "all" else [ASR]
if MONO_LIMIT < 0 or MONO_MAX_NEW_TOKENS < 1:
    raise ValueError("MONO_LIMIT은 0 이상, MONO_MAX_NEW_TOKENS는 1 이상이어야 합니다.")
print("우리 시스템:", OURS_SELECTED, "| 전사기:", ASR_LIST if OURS_SELECTED else "-", "| 단일채널 비교군:", MONO_SELECTED)

## 개인 Drive 연결 (처음 한 번)

학교 계정의 Colab Pro+를 유지하고 개인 계정의 Drive를 **Google 공식 Python 라이브러리**로 사용합니다.
[개인 Drive 설정 가이드](https://github.com/hd1534/variable-channel-spatial-asr/blob/main/docs/personal-drive.md)에 따라
Google Cloud 프로젝트에서 Drive API와 Desktop OAuth 클라이언트를 만들고, Mac에서 `python3 setup_drive.py`를 실행하세요.
생성된 `runs/private/drive-auth.json` 내용을 **Colab 왼쪽 열쇠 아이콘 → 보안 비밀 → `DRIVE_AUTH`**에 저장하고 노트북 접근을 허용합니다.
인증 파일·토큰은 채팅, 코드 셀, 출력, Git에 붙여 넣지 않습니다. 기존 rclone 설정은 사용하지 않습니다.

`STORAGE_BACKEND="drive_api"`가 기본입니다. 학교 Drive를 마운트하지 않으며, 파일은 VM의 `LOCAL_STORE_ROOT`에서 읽고 씁니다.
재접속하면 마지막 완료 백업을 검증해 복원합니다. 같은 계정의 일반 Drive 마운트는 `mounted_drive`로 선택할 수 있습니다.
OAuth 앱이 **Testing** 상태이면 갱신 토큰이 7일 뒤 만료되므로, 장기 사용 전 가이드의 게시 상태 안내도 확인하세요.

여러 런타임이 같은 백업 폴더를 동시에 써도 됩니다. 학습 노트북이 다른 런타임에서 돌고 있어도 이 노트북을 같은 `DRIVE_AUTH`로 연결할 수 있습니다.
런타임마다 자기 백업 이력을 이어 가고, 연결 셀과 명령이 시작할 때마다 다른 런타임의 이력과 합칩니다. 그래서 학습 노트북이 나중에 백업한 `runs/` 결과도 다음 명령이 시작할 때 받아 옵니다.
한 런타임만 바꾼 폴더는 그 런타임의 버전을 씁니다. 같은 `eval/<이름>`이나 `mono_baselines/<실행>` 폴더를 두 런타임이 모두 바꿨다면 파일을 더 나중에 쓴 버전을 쓰고, 다른 버전은 백업 이력에 남깁니다(가이드의 `export --commit`으로 꺼냅니다). ASR 전사 캐시는 두 런타임의 실행 계약과 세션 오디오 해시가 같으면 양쪽을 합치고, 다르면 섞지 않고 파일을 더 나중에 쓴 쪽의 캐시 폴더 전체를 씁니다. 같은 시스템을 두 런타임에서 동시에 채점하지는 마세요.
**이전 코드에서 올릴 때**: 이전 커밋(`9123e53`)의 코드로 돌고 있는 런타임은 다른 런타임이 백업을 올리면 다음 주기 백업에서 `remote advanced in another runtime` 오류로 작업을 멈춥니다. 새 코드의 연결 셀은 연결하자마자 백업을 올리므로, 새 코드로 다른 런타임을 연결하기 전에 이전 코드로 돌고 있는 런타임을 모두 멈추거나 새 코드로 다시 실행하세요.

`RESTORE_SKIP`은 이 VM으로 내려받지 않을 경로 접두어 목록입니다. 기본값(`None`)은 평가에 쓰지 않는 `cache`(LibriSpeech), `rirbanks`, 학습 재개용 체크포인트(`runs/<실행>/ckpt`)와, 고르지 않은 시스템만 쓰는 가중치를 건너뜁니다. `[]`이면 모두 받습니다. 건너뛴 파일도 백업에는 그대로 남습니다.

120초마다 결과를 백업하고, 명령 종료 시 대용량 모델 가중치도 백업합니다.
업로드한 크기·체크섬을 확인한 뒤에만 완료 기록을 저장합니다. 전송 실패는 셀 오류로 표시합니다. 연결 셀이 실패하면 이후 명령 셀도 백업 없이 실행하지 않고 오류로 멈춥니다.
갑작스러운 런타임 종료 시 **마지막 완료 백업 이후 작업은 유실될 수 있습니다**. 마지막 셀의 `개인 Drive 최종 백업 완료`까지 확인하세요.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from typing import Any

if "storage" not in globals():
    storage: Any | None = None
elif storage is not None:
    try:
        storage.close()
    except RuntimeError as exc:
        print(f"이전 백업 세션 종료: {exc}", flush=True)
    storage = None


def stream(command, shell: bool = False) -> None:
    """Run a command and show its output in the cell (Colab hides output a child writes directly)."""
    proc = subprocess.Popen(command, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise subprocess.CalledProcessError(proc.returncode, command)


def run_local(arguments: list[str]) -> None:
    """Run a command without a backup session: git/pip before the connection exists, or mounted_drive mode."""
    print("$", " ".join(arguments), flush=True)
    stream(arguments)


def require_backup() -> None:
    """Refuse to keep working without backups in drive_api mode (for example after a failed reconnect)."""
    if STORAGE_BACKEND == "drive_api" and storage is None:
        raise RuntimeError("개인 Drive 백업 세션이 없습니다. 연결 셀의 오류를 해결한 뒤 그 셀부터 다시 실행하세요.")


def run_command(arguments: list[str]) -> None:
    """Run a command without shell interpolation and stop on failure."""
    require_backup()
    if storage is None:
        run_local(arguments)
        return
    print("$", " ".join(arguments), flush=True)
    storage.run(arguments)


def sh(command: str) -> None:
    """Run a shell command through the backup session and stop on failure."""
    require_backup()
    print("$", command, flush=True)
    if storage is None:
        stream(command, shell=True)
        return
    storage.run(command, shell=True)


def drop_repo_modules(repo_dir: str) -> int:
    """Forget modules imported from the repository so later imports use the checked-out code."""
    root = Path(repo_dir).resolve()
    stale: list[str] = []
    for name, module in list(sys.modules.items()):
        try:
            paths = [getattr(module, "__file__", None), *(getattr(module, "__path__", None) or [])]
            inside = any(isinstance(p, str) and os.path.isabs(p) and Path(p).resolve().is_relative_to(root)
                         for p in paths)
        except Exception:  # special modules whose attributes cannot be read are not repository modules
            continue
        if inside and name != "__main__":
            stale.append(name)
    for name in stale:
        del sys.modules[name]
    return len(stale)


# The backup session is connected below, so git and pip run through run_local before it.
if not Path(REPO_DIR, ".git").is_dir():
    run_local(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
               REPO_URL, REPO_DIR])
run_local(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/*", "!/data/eval-dev/audio/"])
run_local(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", COMMIT or BRANCH])
run_local(["git", "-C", REPO_DIR, "checkout", "FETCH_HEAD"])
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
run_local(["git", "log", "-1", "--format=%h %s"])
# A re-run in the same kernel must not keep modules imported from the previous commit.
print(f"이전 코드로 불러온 프로젝트 모듈 {drop_repo_modules(REPO_DIR)}개를 비웠습니다.", flush=True)

if STORAGE_BACKEND == "drive_api":
    from drive_storage import connect_personal_drive, model_cache_dirs
    run_local([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-drive.txt"])
    if RESTORE_SKIP is None:
        # Evaluation reads runs/<run>/{config.json,summary.json,best.pt}; corpora, RIR banks and resume checkpoints stay behind.
        RESTORE_SKIP = ["cache", "rirbanks"] + [f"runs/{name}{RUN_SUFFIX}/ckpt" for name in OURS]
        unused: list[str] = []
        if not OURS_SELECTED:
            RESTORE_SKIP += ["runs", "models/ecapa"]
        if not OURS_SELECTED or "wav2vec2" not in ASR_LIST:
            unused.append("facebook/wav2vec2-base-960h")
        if not (OURS_SELECTED and "qwen3-1.7b" in ASR_LIST) and "nemotron" not in MONO_SELECTED:
            unused.append("Qwen/Qwen3-ASR-1.7B")
        for name, repos in (("nemotron", ["nvidia/Nemotron-3-Diarization"]),
                            ("moss", ["OpenMOSS-Team/MOSS-Transcribe-Diarize"]),
                            ("vibevoice", ["microsoft/VibeVoice-ASR", "Qwen/Qwen2.5-7B"])):  # Qwen2.5-7B: VibeVoice tokenizer
            if name not in MONO_SELECTED:
                unused += repos
        RESTORE_SKIP += model_cache_dirs(unused)
    print("복원 제외 (백업에는 남음):", RESTORE_SKIP, flush=True)
    storage = connect_personal_drive(Path(LOCAL_STORE_ROOT), secret_name=DRIVE_SECRET,
                                     interval_sec=BACKUP_EVERY_SEC, skip=RESTORE_SKIP)
elif STORAGE_BACKEND == "mounted_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive
        drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
else:
    raise ValueError("STORAGE_BACKEND를 확인하세요.")
print(f"저장소: {STORAGE_BACKEND} | 작업 경로: {DRIVE_ROOT}", flush=True)

Path(DRIVE_ROOT, "mono_baselines").mkdir(parents=True, exist_ok=True)

os.environ["VCSASR_MODEL_CACHE"] = str(Path(DRIVE_ROOT, "models", "asr"))
# Cache public Hugging Face weights without moving authentication tokens into the backup tree.
os.environ["HF_HUB_CACHE"] = str(Path(DRIVE_ROOT, "models", "huggingface"))


def mono_run_name(name: str) -> str:
    """Select a separate output directory when experiment settings change."""
    return f"{name}-{MONO_RUN_TAG}" if MONO_RUN_TAG else name

## 실행 환경 확인과 설치

GPU를 확인하고, 고른 시스템에 필요한 것만 설치합니다.
- 우리 시스템: 화자 임베딩용 `speechbrain`, `wav2vec2` 가중치, Qwen3-ASR 전용 venv
- 단일채널 비교군: 모델마다 별도 venv에 공식 구현

기존 학습 환경은 갱신하지 않습니다. 모델 가중치와 공식 소스는 코드에 기록된 revision을 쓰고, 설치된 패키지 목록은 `mono_baselines/runtimes/`에 남깁니다.

In [ ]:
import shutil
import torch
from setup_asr_env import setup

if not torch.cuda.is_available():
    raise RuntimeError("런타임 유형을 GPU로 바꾼 뒤 다시 실행하세요.")
GPU: str = torch.cuda.get_device_name(0)
VRAM_GB: float = torch.cuda.get_device_properties(0).total_memory / 1e9
print(GPU, "| VRAM GB", round(VRAM_GB, 1), "| compute capability", torch.cuda.get_device_capability())
if "vibevoice" in MONO_SELECTED and VRAM_GB < 20:
    raise RuntimeError(f"VibeVoice는 가중치만 약 18GB라 이 GPU({GPU}, {VRAM_GB:.0f}GB)에서 돌지 않습니다. "
                       "런타임 유형에서 A100을 고르거나 SYSTEMS에서 vibevoice를 빼세요.")

if OURS_SELECTED:
    sh("pip freeze | grep -iE '^(torch|torchaudio|numpy)==' > /content/constraints.txt")
    sh('pip -q install -c /content/constraints.txt "speechbrain==1.1.2"')
    if "wav2vec2" in ASR_LIST:
        sh("make asr-cache")

profiles: list[str] = list(dict.fromkeys(
    MONO_SELECTED + (["qwen"] if "nemotron" in MONO_SELECTED or (OURS_SELECTED and "qwen3-1.7b" in ASR_LIST) else [])))
interpreters: dict[str, str] = {name: str(setup(name, Path(ENV_ROOT))) for name in profiles}
for name in profiles:
    destination: Path = Path(DRIVE_ROOT, "mono_baselines", "runtimes", name)
    destination.mkdir(parents=True, exist_ok=True)
    for filename in ("resolved-requirements.txt", "vcsasr-runtime.json"):
        shutil.copy2(Path(ENV_ROOT, name, filename), destination / filename)

## 1. 우리 시스템 본 평가

학습 노트북이 남긴 `runs/<시스템>`마다 두 평가셋에서 추론하고, 전사기별로 전사·채점합니다.
- **임계값**: 비교군은 검증 세트에서 고른 임계값(단순 결합은 가중치도)을 그대로 씁니다. 제안 모델은 `best.pt`와 그 임계값을 씁니다. 본 평가셋에서는 아무것도 조정하지 않습니다.
- **제안 모델 변형**: 전체 모델(`full`) 외에 진단 변형 두 개도 함께 채점합니다.
  - `learned_space`: 학습된 공간 경로만 쓴 것
  - `fixed_gate`: 게이트를 마이크 수별 고정 가중치로 바꾼 것
- **단순 결합**: 목소리 전용과 공간 전용의 평가 임베딩이 있으면 그것을 그대로 합쳐 씁니다.
- **학습이 덜 끝난 시스템**: 건너뜁니다. 학습 노트북에서 마저 학습한 뒤 이 셀을 다시 실행하세요.

In [ ]:
import eval_t2
import numpy as np
from training.config import load_config_dict
from training.engine import run_is_complete


def eval_run_name(base_name: str, asr_backend: str) -> str:
    """Return an ASR-specific evaluation directory name."""
    return f"{base_name}-{asr_backend}"


def migrate_legacy_eval_dir(base_name: str) -> None:
    """Move a pre-suffix eval/<run_name> directory to eval/<run_name>-<asr> if present."""
    legacy = Path(DRIVE_ROOT, "eval", base_name)
    if not legacy.is_dir():
        return
    detected_asr = "qwen3-1.7b"
    for score_file in sorted(legacy.glob("*/*/score.json")):
        try:
            info = json.loads(score_file.read_text(encoding="utf-8")).get("eval_t2_run") or {}
            if "wav2vec2" in str(info.get("asr", "")):
                detected_asr = "wav2vec2"
                break
        except Exception:
            pass
    target = Path(DRIVE_ROOT, "eval", eval_run_name(base_name, detected_asr))
    if not target.exists():
        legacy.rename(target)
        print(f"[migrate] 기존 평가 폴더 {legacy.name} -> {target.name} 이동 완료", flush=True)


def find_component_eval_dir(prefix: str, asr_backend: str) -> Path | None:
    """Find an eval directory for prefix that already has embeddings for all sets."""
    candidates = [
        Path(DRIVE_ROOT, "eval", eval_run_name(prefix, asr_backend)),
        Path(DRIVE_ROOT, "eval", eval_run_name(prefix, "qwen3-1.7b")),
        Path(DRIVE_ROOT, "eval", eval_run_name(prefix, "wav2vec2")),
        Path(DRIVE_ROOT, "eval", prefix),
    ]
    for cand in candidates:
        if all((cand / s / "embeddings.npz").is_file() for s in eval_t2.SETS):
            return cand
    return None


def share_embeddings_across_asr(base_name: str, target_out: Path, cfg, best_pt: Path | None,
                                comp_dirs: list[str] | None) -> None:
    """Reuse verified embeddings.npz from a sibling ASR directory so each system embeds only once."""
    siblings = [
        Path(DRIVE_ROOT, "eval", eval_run_name(base_name, "qwen3-1.7b")),
        Path(DRIVE_ROOT, "eval", eval_run_name(base_name, "wav2vec2")),
        Path(DRIVE_ROOT, "eval", base_name),
    ]
    for set_name in eval_t2.SETS:
        set_dir = Path(REPO_DIR, "data", set_name)
        expected_fp = eval_t2.inputs_fingerprint(cfg, set_dir, best_pt, 0, comp_dirs)
        dst = target_out / set_name / "embeddings.npz"
        if dst.is_file():
            try:
                with np.load(dst, allow_pickle=False) as z:
                    if "fingerprint" in z and str(z["fingerprint"]) == expected_fp:
                        continue
            except Exception:
                pass
        for sib in siblings:
            src = sib / set_name / "embeddings.npz"
            if src == dst or not src.is_file():
                continue
            try:
                with np.load(src, allow_pickle=False) as z:
                    if "fingerprint" in z and str(z["fingerprint"]) == expected_fp:
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        tmp = dst.with_name(dst.name + ".part")
                        shutil.copy2(src, tmp)
                        os.replace(tmp, dst)
                        break
            except Exception:
                pass



def run_config(run_dir: Path):
    """The configuration the training notebook recorded for this run (None before it wrote one)."""
    path = run_dir / "config.json"
    if not path.is_file():
        return None
    return load_config_dict(json.loads(path.read_text(encoding="utf-8"))["config"])


asr_cache_dir: str = str(Path(DRIVE_ROOT, ASR_CACHE_NAME))
not_ready: list[str] = []
for name in OURS_SELECTED:
    run_name = name + RUN_SUFFIX
    run_dir = Path(DRIVE_ROOT, "runs", run_name)
    cfg = run_config(run_dir)
    if cfg is None or not run_is_complete(run_dir, cfg):
        print(f"[대기] {run_name}: 학습·검증이 끝난 summary.json이 아직 없습니다. 학습 노트북에서 끝낸 뒤 다시 실행하세요.", flush=True)
        not_ready.append(run_name)
        continue
    migrate_legacy_eval_dir(run_name)
    summary = json.loads(Path(run_dir, "summary.json").read_text(encoding="utf-8"))
    best_pt = Path(run_dir, "best.pt") if cfg.model.name == "proposed" else None
    for asr_backend in ASR_LIST:
        eval_name = eval_run_name(run_name, asr_backend)
        eval_out = Path(DRIVE_ROOT, "eval", eval_name)
        comp_dirs: list[str] | None = None
        comp_args: list[str] = []
        if cfg.model.name == "naive_fusion":
            v_eval = find_component_eval_dir("voice_only" + RUN_SUFFIX, asr_backend)
            s_eval = find_component_eval_dir("space_only" + RUN_SUFFIX, asr_backend)
            if v_eval is not None and s_eval is not None:
                comp_dirs = [str(v_eval), str(s_eval)]
                comp_args = ["--components-from", *comp_dirs]
        share_embeddings_across_asr(run_name, eval_out, cfg, best_pt, comp_dirs)
        if eval_t2.eval_is_complete(eval_out, cfg, summary, best=best_pt, components_from=comp_dirs,
                                    asr=asr_backend, asr_cache=asr_cache_dir):
            print(f"[skip] {eval_name}: 현재 summary.json / best.pt와 일치하는 본 평가 결과가 이미 있습니다.", flush=True)
            continue
        print(f">>> [{eval_name}] 본 평가셋(eval-t2, eval-t2-static) 추론 및 채점 (ASR={asr_backend})", flush=True)
        arguments: list[str] = [sys.executable, "eval_t2.py", "--run-dir", str(run_dir), "--out", str(eval_out),
                                "--device", "cuda", "--model-cache", str(Path(DRIVE_ROOT, "models", "ecapa")),
                                "--asr", asr_backend, "--asr-cache", asr_cache_dir, *comp_args]
        if asr_backend == "qwen3-1.7b":
            arguments += ["--asr-python", interpreters["qwen"], "--asr-device", "cuda"]
        run_command(arguments)

## 2. 최신 단일채널 비교군 (Track B 포함)

완료한 세션은 건너뜁니다. 모델·디코딩 설정·오디오가 달라지면 기존 결과와 섞지 않고 멈추므로, 바뀐 실험은 다른 출력 경로를 쓰세요. `MONO_LIMIT=2`로 확인한 뒤 `0`으로 바꾸면 나머지 세션을 이어서 처리합니다. 파싱 실패나 토큰 상한에 도달한 출력은 성공으로 저장하지 않습니다.

모델·토큰 상한을 바꿀 때에는 `MONO_RUN_TAG`로 새 실험 경로를 지정합니다. CPU/GPU 정밀도나 패키지 버전이 달라졌다면 `ASR_CACHE_NAME`도 새 이름으로 바꾸세요.

Nemotron은 우리 시스템과 같은 Qwen 전사·최소 1초 slot-run 규칙을 쓰고, VibeVoice/MOSS는 자체 전사를 씁니다. 예측 화자가 2명을 넘으면 발화 시간이 긴 두 화자를 유지하고 나머지는 가장 긴 화자에 합칩니다(`top2-duration-fold-v1`). 이 후처리는 정답을 쓰지 않으며 결과에 정책을 기록합니다.

In [ ]:
mono_failed: dict[str, str] = {}
for name in MONO_SELECTED:
    arguments: list[str] = [sys.executable, "mono_baselines.py", "--backend", name,
                            "--python", interpreters[name], "--out", str(Path(DRIVE_ROOT, "mono_baselines", mono_run_name(name))),
                            "--asr-cache", str(Path(DRIVE_ROOT, ASR_CACHE_NAME)), "--device", "cuda",
                            "--limit", str(MONO_LIMIT), "--max-new-tokens", str(MONO_MAX_NEW_TOKENS), "--sets", *SETS]
    if name == "nemotron":
        arguments += ["--qwen-python", interpreters["qwen"]]
    try:
        run_command(arguments)
    except subprocess.CalledProcessError as exc:
        # 한 비교군이 실패해도 나머지는 계속 돈다. 원인은 위 출력과 아래 요약에 있다.
        mono_failed[name] = str(exc)
        print(f"[실패] {name}: 나머지 비교군을 계속합니다.", flush=True)
if mono_failed:
    raise RuntimeError("실패한 단일채널 비교군 (끝난 작업은 다시 실행하면 건너뜁니다):\n\n"
                       + "\n\n".join(f"{name}:\n{message}" for name, message in mono_failed.items()))

## 3. 결과표

`eval/`의 우리 시스템과 `mono_baselines/`의 단일채널 비교군을 함께 보여 줍니다.
- 열 순서는 M = 4 → 3 → 2 → 1입니다. 단일채널 비교군의 M은 짝지은 우리 시스템의 마이크 수이고, 실제 입력은 언제나 1ch입니다.
- 같은 마이크의 출력을 서로 다른 M에서 반복 평가한 값을 독립 표본으로 보지 마세요. 신뢰구간은 세션 단위로 묶어 계산합니다.
- `asr_comparison.json`에는 같은 케이스끼리 짝지은 비교표가 들어갑니다. 평가 부분집합이 다르면 별도 population으로 표시합니다.

In [ ]:
eval_root = Path(DRIVE_ROOT, "eval")
for set_name in SETS:
    print(f"=== [{set_name}] causal, 우리 시스템 (Spk Acc / cpWER) ===")
    print(f"{'run/variant':<46}" + "".join(f"{'M=' + M + ' Acc':>10}{'cpWER':>8}" for M in ("4", "3", "2", "1")))
    for score_path in sorted(eval_root.glob(f"*/{set_name}/*/score.json")):
        rep = json.loads(score_path.read_text(encoding="utf-8"))
        by_n = rep.get("breakdown", {}).get("by_n_channels", {})
        cells = []
        for M in ("4", "3", "2", "1"):
            met = (by_n.get(M) or {}).get("metrics", {})
            cells.append(f"{met.get('speaker_attribution_accuracy', float('nan')):>10.3f}{met.get('cpwer', float('nan')):>8.3f}")
        print(f"{score_path.parents[2].name + '/' + score_path.parent.name:<46}" + "".join(cells))
    print()

mono_reports: dict[str, dict] = {}
for report_path in sorted(Path(DRIVE_ROOT, "mono_baselines").glob("*/score.json")):
    mono_reports[report_path.parent.name] = json.loads(report_path.read_text(encoding="utf-8"))
for set_name in SETS:
    if not mono_reports:
        break
    print(f"=== [{set_name}] full_audio, 단일채널 비교군 (input=1ch, paired M) ===")
    for name, report in mono_reports.items():
        values = [f"M={m}: Acc {met['speaker_attribution_accuracy']:.3f}, cpWER {met['cpwer']:.3f}"
                  for m, met in sorted(report.get(set_name, {}).get("by_comparison_M", {}).items(), reverse=True)]
        limit = (report.get("baseline") or {}).get("limit")
        print(f"{name:<20}", " | ".join(values), f"(limit={limit})" if limit else "")
    print()

if OURS_SELECTED and not_ready:
    print("학습이 끝나지 않아 건너뛴 시스템:", not_ready)

run_command([sys.executable, "compare_asr.py", "--eval-root", str(eval_root),
             "--mono-root", str(Path(DRIVE_ROOT, "mono_baselines")),
             "--out", str(Path(DRIVE_ROOT, "asr_comparison.json")), "--sets", *SETS])

## 4. 전체 결과 블록 — 이 출력을 그대로 복사해 붙여 넣어 주세요

Drive에 있는 **모든 본 평가 결과**(`eval/`, `mono_baselines/`)와 학습 노트북의 검증 요약(`runs/*/summary.json`)을 한 번에 출력합니다.

In [ ]:
from train import SUMMARY_KEYS

all_runs = {}
for path in sorted(Path(DRIVE_ROOT, "runs").glob("*/summary.json")):
    s = json.loads(path.read_text(encoding="utf-8"))
    all_runs[path.parent.name] = {k: s.get(k) for k in SUMMARY_KEYS if s.get(k) is not None}

all_evals = {}
eval_root = Path(DRIVE_ROOT, "eval")
if eval_root.is_dir():
    for run_path in sorted(eval_root.iterdir()):
        if not run_path.is_dir():
            continue
        run_res = {}
        for set_name in ("eval-t2", "eval-t2-static"):
            for score_path in sorted((run_path / set_name).glob("*/score.json")):
                rep = json.loads(score_path.read_text(encoding="utf-8"))
                key = f"{set_name}/{score_path.parent.name}"
                run_res[key] = {
                    "overall": {k: rep["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")},
                    "by_M": {m: {k: v["metrics"].get(k) for k in ("speaker_attribution_accuracy", "cpwer", "wer")}
                             for m, v in rep.get("breakdown", {}).get("by_n_channels", {}).items()},
                    "frame_optimal_spk_acc_by_M": rep.get("frame_optimal_spk_acc_by_M"),
                    "eval_t2_run": rep.get("eval_t2_run"),
                }
        if run_res:
            all_evals[run_path.name] = run_res

comparison_path = Path(DRIVE_ROOT, "asr_comparison.json")
print("===== VC-SASR BENCHMARK RESULTS (paste this back) =====")
print(json.dumps({"validation_runs": all_runs, "eval_t2_runs": all_evals, "mono_baselines": mono_reports,
                  "asr_comparison": json.loads(comparison_path.read_text(encoding="utf-8")) if comparison_path.is_file() else None},
                 ensure_ascii=False))
print("===== END =====")
require_backup()
if storage is not None:
    storage.flush()
    print("개인 Drive 최종 백업 완료", flush=True)